# Popularity bias in recommenders

Cleans Amazon Reviews 2023 data, splits it by time, then trains and evaluates four recommenders
(`mostpop`, `itemcf`, `mf`, `gru`) on accuracy and popularity-bias metrics.

Run this notebook from the project folder. Data: https://amazon-reviews-2023.github.io/

In [ ]:
%pip install -q -r requirements.txt

## Settings

In [ ]:
CATEGORY = "Clothing_Shoes_and_Jewelry"
RAW_REVIEWS = f"data/{CATEGORY}.jsonl.gz"
K_CORE = 2
MODELS = ["mostpop", "itemcf", "mf", "gru"]
SAMPLE_EVERY = None   # e.g. 50 keeps 1 in 50 users for a quick run

## Download the raw reviews
Downloads `<CATEGORY>.jsonl.gz` from the Amazon Reviews 2023 site into `data/` (Clothing is ~7 GB).
Skipped if the raw file or the cleaned parquet already exists.

In [ ]:
import os
import urllib.request

URL = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/{CATEGORY}.jsonl.gz"

def progress(n, size, total):
    if n % 12_000 == 0:  # about every 100 MB
        print(f"\r{n * size / 1e9:.1f} / {total / 1e9:.1f} GB", end="")

if os.path.exists(f"data/{CATEGORY}_interactions.parquet") or os.path.exists(RAW_REVIEWS):
    print("Already have the data, skipping download")
else:
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(URL, RAW_REVIEWS + ".part", progress)  # .part so a failed download isn't mistaken for a full one
    os.replace(RAW_REVIEWS + ".part", RAW_REVIEWS)
    print("\nDone")

## Clean the raw reviews
Skip this if `data/<CATEGORY>_interactions.parquet` already exists.

In [ ]:
!python prepare.py {RAW_REVIEWS} {CATEGORY}

## Optional: sample users

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

if SAMPLE_EVERY:
    parts = []
    for batch in pq.ParquetFile(f"data/{CATEGORY}_interactions.parquet").iter_batches(batch_size=500_000):
        df = batch.to_pandas()
        parts.append(df[pd.util.hash_pandas_object(df.user_id, index=False).to_numpy() % SAMPLE_EVERY == 0])
    CATEGORY = f"{CATEGORY}_sample"
    pd.concat(parts, ignore_index=True).to_parquet(f"data/{CATEGORY}_interactions.parquet", index=False)

## Split into train / validation / test

In [ ]:
!python split.py {CATEGORY} {K_CORE}

## Train and evaluate

In [ ]:
for m in MODELS:
    !python run.py {CATEGORY} {K_CORE} {m}

## Results

In [ ]:
results = pd.read_csv("results.csv")
results[results.category == CATEGORY]